# 03 — Baseline Modeling

**Project:** Predicting 30-Day Hospital Readmission  
**Phase:** 3 — Baseline Models & Evaluation

This notebook trains three baseline classifiers on the processed feature dataset, evaluates them on the held-out validation set, and produces a full suite of diagnostic plots.

| # | Section |
|---|---|
| 1 | Environment setup |
| 2 | Load config, features, and split |
| 3 | Train baseline models |
| 4 | Cross-validation results |
| 5 | Validation-set evaluation |
| 6 | Metrics comparison table |
| 7 | ROC and PR curves |
| 8 | Confusion matrix — best model |
| 9 | Calibration curves |
| 10 | Feature importances |
| 11 | Evaluation on test set |
| 12 | Save artifacts |
| 13 | Interpretation and next steps |

---
### ⚠️ Dataset note
The dataset has a **74% positive (readmitted) rate** — the majority class is readmitted, the inverse of real-world clinical data (~15–25%).  
Because of this:
- **PR-AUC must be interpreted relative to the 0.74 no-skill baseline**, not 0.50.
- **ROC-AUC is the cleaner discrimination metric** for understanding true predictive power.
- Class-weighted training is applied to all models.

---
## 1 — Environment setup

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Anchor all relative paths to the project root (one level up from notebooks/)
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, get_logger, set_seed, save_model
from src.modeling import (
    load_features, make_splits, build_baselines,
    cross_validate_model, evaluate,
    plot_roc_curves, plot_pr_curves, plot_confusion_matrix,
    plot_calibration_curves, plot_feature_importance,
    compare_models, select_best_model,
)

logger = get_logger('03_modeling_baseline')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

print(f'Project root : {ROOT}')
print('Environment ready.')

---
## 2 — Load config, features, and split

In [ ]:
config = load_config(ROOT / 'config' / 'config.yaml')
set_seed(config['random_seed'])

# Inject project root so modeling functions can resolve relative paths correctly
# when the notebook is launched from the notebooks/ subdirectory.
config['_base_dir'] = str(ROOT)

TARGET   = config['data']['target_column']
SEED     = config['random_seed']
EXCLUDED = config.get('model', {}).get('exclude_columns', [])

print(f'Target          : {TARGET}')
print(f'Random seed     : {SEED}')
print(f'Excluded columns: {EXCLUDED}')

In [ ]:
# base_dir=ROOT anchors the relative features_data path to the project root
X, y, feat_names = load_features(config, base_dir=ROOT)

print(f'Feature matrix : {X.shape}')
print(f'Class balance  : {y.value_counts().to_dict()}')
print(f'Positive rate  : {y.mean():.1%}')

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(X, y, config)

print(f'Train : {X_train.shape[0]:,} rows  (pos rate {y_train.mean():.1%})')
print(f'Val   : {X_val.shape[0]:,} rows  (pos rate {y_val.mean():.1%})')
print(f'Test  : {X_test.shape[0]:,} rows  (pos rate {y_test.mean():.1%})')

---
## 3 — Train baseline models

Three estimators are trained:
- **Logistic Regression** — linear baseline; `class_weight='balanced'`
- **Random Forest** — non-linear ensemble; `class_weight='balanced'`
- **Gradient Boosting** — XGBoost if installed, else `HistGradientBoostingClassifier`; `class_weight='balanced'`

All hyperparameters are read from `config/config.yaml`.

In [ ]:
baselines = build_baselines(config)
print('Baseline estimators:')
for name, model in baselines.items():
    print(f'  {name}: {type(model).__name__}')

---
## 4 — Cross-validation (training set)

5-fold stratified CV on the **training set** gives an unbiased estimate of generalisation  
before the validation set is ever touched.

In [ ]:
cv_summaries = {}
fitted = {}

for name, model in baselines.items():
    print(f'Cross-validating: {name} ...')
    cv_res = cross_validate_model(model, X_train, y_train, config)
    cv_summaries[name] = {
        k[5:]: (v.mean(), v.std())
        for k, v in cv_res.items() if k.startswith('test_')
    }
    model.fit(X_train, y_train)   # refit on full train set
    fitted[name] = model
    print()

print('All models fitted.')

In [ ]:
cv_rows = []
for name, metrics in cv_summaries.items():
    row = {'model': name}
    for metric, (mean, std) in metrics.items():
        row[metric] = f'{mean:.4f} ± {std:.4f}'
    cv_rows.append(row)

cv_df = pd.DataFrame(cv_rows).set_index('model')
print('Cross-validation results (mean ± std across 5 folds):')
display(cv_df)

---
## 5 — Validation-set evaluation

In [ ]:
eval_results = {}
for name, model in fitted.items():
    eval_results[name] = evaluate(model, X_val, y_val)

for name, m in eval_results.items():
    print(
        f'{name:<25}  ROC-AUC={m["roc_auc"]:.4f}  '
        f'PR-AUC={m["pr_auc"]:.4f}  F1={m["f1"]:.4f}  Brier={m["brier"]:.4f}'
    )

---
## 6 — Metrics comparison table

**Primary metrics:** ROC-AUC and PR-AUC  
**Secondary:** Accuracy, Precision, Recall, F1, Brier Score  

> **How to read PR-AUC here:** The no-skill baseline equals the positive prevalence (~0.74).  
> Compare the **lift** column — the actual improvement over random — not the absolute PR-AUC.

In [ ]:
summary = compare_models(eval_results)
noskill = float(y_val.mean())
summary.insert(2, 'pr_auc_lift', (summary['pr_auc'] - noskill).round(4))

print(f'No-skill PR-AUC baseline (positive prevalence): {noskill:.4f}\n')
display(summary)

In [ ]:
metrics_to_plot = ['roc_auc', 'pr_auc', 'f1', 'brier']
plot_df = (
    summary[metrics_to_plot]
    .reset_index()
    .melt(id_vars='model', var_name='metric', value_name='score')
)
palette = sns.color_palette('muted', len(fitted))

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(14, 4))
for ax, metric in zip(axes, metrics_to_plot):
    sub = plot_df[plot_df['metric'] == metric]
    bars = ax.bar(sub['model'], sub['score'], color=palette)
    ax.set_title(metric.upper().replace('_', '-'), fontsize=10)
    ax.set_ylim(0, 1.08)
    ax.tick_params(axis='x', rotation=18, labelsize=8)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=8,
        )
    if metric == 'pr_auc':
        ax.axhline(noskill, color='red', linestyle='--', lw=1.2, label=f'No-skill ({noskill:.2f})')
        ax.legend(fontsize=7)
    if metric == 'roc_auc':
        ax.axhline(0.5, color='red', linestyle='--', lw=1.2, label='Random (0.50)')
        ax.legend(fontsize=7)

plt.suptitle('Baseline Model Comparison — Validation Set', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'metrics_comparison_baselines.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
best_name = select_best_model(summary, metric='pr_auc')
best_model = fitted[best_name]

print(f'Best baseline model (by PR-AUC): {best_name}\n')
for k, v in eval_results[best_name].items():
    print(f'  {k:<12}: {v:.4f}')

---
## 7 — ROC and PR curves

**ROC curve:** Discrimination across all thresholds.  AUC 0.5 = random, 1.0 = perfect.  
**PR curve:** Precision-Recall trade-off — more informative under class imbalance.  
The dashed red line marks the no-skill PR-AUC (≈ positive prevalence = 0.74).

In [ ]:
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pal = sns.color_palette('tab10', len(fitted))
prevalence = float(y_val.mean())

# ROC
ax = axes[0]
for (name, model), color in zip(fitted.items(), pal):
    prob = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, prob)
    ax.plot(fpr, tpr, label=f'{name}  AUC={roc_auc_score(y_val, prob):.3f}',
            color=color, lw=2)
ax.plot([0, 1], [0, 1], 'r--', lw=1, label='Random (0.500)')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
       title='ROC Curves — Validation Set')
ax.legend(fontsize=8)

# PR
ax = axes[1]
for (name, model), color in zip(fitted.items(), pal):
    prob = model.predict_proba(X_val)[:, 1]
    prec, rec, _ = precision_recall_curve(y_val, prob)
    ax.plot(rec, prec,
            label=f'{name}  PR-AUC={average_precision_score(y_val, prob):.3f}',
            color=color, lw=2)
ax.axhline(prevalence, color='red', linestyle='--', lw=1.2,
           label=f'No-skill ({prevalence:.3f})')
ax.set(xlabel='Recall', ylabel='Precision',
       title='Precision-Recall Curves — Validation Set')
ax.legend(fontsize=8)

plt.suptitle('ROC and PR Curves', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'roc_pr_curves_baselines.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 8 — Confusion matrix — best model

Default threshold = 0.5.  With a 74% positive prevalence, lowering the threshold would  
improve recall (sensitivity) at the cost of precision — a clinical trade-off.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

THRESHOLD = 0.5
y_prob_best = best_model.predict_proba(X_val)[:, 1]
y_pred_best = (y_prob_best >= THRESHOLD).astype(int)

cm = confusion_matrix(y_val, y_pred_best)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    cm, display_labels=['Not Readmitted (0)', 'Readmitted (1)']
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}\n(threshold={THRESHOLD}, validation set)')
plt.tight_layout()
plt.savefig(FIGURES / 'confusion_matrix_best_baseline.png', bbox_inches='tight', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'Sensitivity (recall) : {tp / (tp + fn):.3f}')
print(f'Specificity          : {tn / (tn + fp):.3f}')
print(f'PPV (precision)      : {tp / (tp + fp):.3f}')

---
## 9 — Calibration curves

Well-calibrated models follow the diagonal — predicted probability = observed frequency.  
Strong deviations indicate the model is over- or under-confident.

In [ ]:
from sklearn.calibration import CalibrationDisplay

fig, ax = plt.subplots(figsize=(7, 6))
pal = sns.color_palette('tab10', len(fitted))
for (name, model), color in zip(fitted.items(), pal):
    CalibrationDisplay.from_estimator(
        model, X_val, y_val, n_bins=10, ax=ax, name=name, color=color
    )
ax.set_title('Calibration Curves — Validation Set')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'calibration_baselines.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 10 — Feature importances

- **Logistic Regression:** |coefficient| (magnitude of linear weights)
- **Random Forest:** Gini impurity decrease (built-in)
- **HistGradientBoosting:** permutation importance on a validation sample  
  (native `feature_importances_` not available; SHAP will be used in Phase 5)

In [ ]:
from sklearn.inspection import permutation_importance

TOP_N = 15
fig, axes = plt.subplots(1, len(fitted), figsize=(18, 6))

# Fix: permutation importance needs a consistent index sample
sample_idx = X_val.sample(1000, random_state=42).index
X_val_sample = X_val.loc[sample_idx]
y_val_sample = y_val.loc[sample_idx]

for ax, (name, model) in zip(axes, fitted.items()):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        xlabel = 'Gini Importance'
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_[0])
        xlabel = '|Coefficient|'
    else:
        perm = permutation_importance(
            model, X_val_sample, y_val_sample,
            n_repeats=5, random_state=42, scoring='roc_auc', n_jobs=-1
        )
        importances = perm.importances_mean
        xlabel = 'Permutation Importance (ROC-AUC)'

    idx = np.argsort(importances)[-TOP_N:]
    ax.barh([feat_names[i] for i in idx], importances[idx],
            color=sns.color_palette('muted')[0])
    ax.set_title(name, fontsize=9)
    ax.set_xlabel(xlabel, fontsize=8)
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle(f'Top {TOP_N} Feature Importances — All Baseline Models', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'feature_importance_all_baselines.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 11 — Test-set evaluation

> The test set is **touched exactly once** — for the final best-model report.  
> It must not be used for model selection, threshold tuning, or feature decisions.

In [ ]:
test_metrics = evaluate(best_model, X_test, y_test)

print(f'Test-set metrics — {best_name}\n')
print(f'  {"Metric":<12}  {"Test":>8}  {"Val":>8}  {"Δ":>8}')
print('  ' + '-' * 38)
for k, v in test_metrics.items():
    val_v = eval_results[best_name][k]
    print(f'  {k:<12}  {v:>8.4f}  {val_v:>8.4f}  {v - val_v:>+8.4f}')

---
## 12 — Save artifacts

In [ ]:
# Best baseline model
model_path = ROOT / config['paths']['model_dir'] / 'best_baseline_model.pkl'
save_model(
    {'name': best_name, 'model': best_model, 'feature_names': feat_names},
    model_path,
)
print(f'Model saved  → {model_path}')

# Metrics CSV (val + test rows)
test_row = pd.DataFrame(
    [test_metrics], index=[f'{best_name} (TEST)']
)
full_summary = pd.concat(
    [summary.drop(columns=['pr_auc_lift'], errors='ignore'), test_row]
)
metrics_path = ROOT / config['paths']['metrics_out']
full_summary.to_csv(metrics_path)
print(f'Metrics saved → {metrics_path}')

print()
print('Figures saved to reports/figures/:')
for f in sorted(FIGURES.glob('*.png')):
    print(f'  {f.name}')

---
## 13 — Interpretation and next steps

### Results summary

| Model | ROC-AUC | PR-AUC | PR-AUC Lift | F1 | Brier |
|---|---|---|---|---|---|
| Logistic Regression | ~0.56 | ~0.78 | ~+0.04 | ~0.64 | ~0.25 |
| Random Forest | ~0.55 | ~0.77 | ~+0.03 | ~0.83 | ~0.22 |
| Gradient Boosting | ~0.54 | ~0.76 | ~+0.02 | ~0.65 | ~0.24 |

### Key findings

**1. ROC-AUC ~0.55 — near-random discrimination**  
All three models sit barely above the random 0.50 baseline. This is the most important finding: the clinical features in this synthetic dataset carry very weak predictive signal for readmission. Real datasets with equivalent features typically yield ROC-AUC 0.65–0.80.

**2. PR-AUC is misleadingly high due to the 74 % positive prevalence**  
The no-skill PR-AUC equals the positive rate (0.74). Every model achieves only ~0.04 lift above this floor. Do not interpret the absolute PR-AUC value — compare `pr_auc_lift` instead.

**3. Random Forest F1 ≈ 0.83 is an artefact, not a strength**  
With class-weighted training and a 74 % majority class, the RF predicts most patients as readmitted, yielding high recall but poor specificity. F1 is inflated because it weights toward the majority class.

**4. Logistic Regression wins by PR-AUC**  
The linear model edges out both tree methods, suggesting the dataset has weak non-linear structure — consistent with synthetic label assignment.

**5. Brier scores near the no-skill benchmark (~0.19)**  
No model improves meaningfully on a naive always-predict-prevalence classifier.

### Leakage columns excluded

| Column | Reason |
|---|---|
| `Followup_Appointment_Scheduled` | Post-discharge decision — not available at prediction time |
| `Medication_Adherence_Score` | Post-discharge observation |

To **test for leakage**, temporarily remove these from `model.exclude_columns` in config and rerun.  
If ROC-AUC jumps significantly (e.g., > 0.70), the columns were leaking label information.

### Columns worth investigating
- `Previous_Readmissions_1Y` — partial overlap with the target for repeated admissions
- `discharge_disposition_cat_*` — redundant with `Discharge_Disposition_*` OHE columns
- `age_group_*` — redundant with continuous `Age`; consider dropping one set

### Next steps

1. **`04_modeling_tuned.ipynb`** — `tune_model()` scaffold is ready; run `RandomizedSearchCV` for all three models  
2. **Calibration** — `calibrate_model()` scaffold is ready; apply after tuning  
3. **Interaction features** — `Age × Comorbidity_Index`, `Severity_Score × Length_of_Stay`  
4. **Feature selection** — drop redundant columns; try `SelectFromModel` or RFECV  
5. **SHAP** — `05_interpretation_error_analysis.ipynb`  
6. **Subgroup analysis** — performance by `age_group`, `Gender`, `Admission_Type`